In [1]:
%cd /kaggle/working
!git clone https://github.com/shadyOg/fyp-dtcr-unet.git || (cd fyp-dtcr-unet && git pull origin main)
%cd /kaggle/working/fyp-dtcr-unet/dtcr-unet
!pip install -q -r requirements.txt


/kaggle/working
Cloning into 'fyp-dtcr-unet'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 180 (delta 88), reused 144 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 2.44 MiB | 12.26 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/kaggle/working/fyp-dtcr-unet/dtcr-unet


In [2]:
%cd /kaggle/working/fyp-dtcr-unet/dtcr-unet
!git pull origin main


/kaggle/working/fyp-dtcr-unet/dtcr-unet
From https://github.com/shadyOg/fyp-dtcr-unet
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
%cd /kaggle/working/fyp-dtcr-unet/dtcr-unet

PROCESSED_DIR = '/kaggle/working/data/processed'

!python data/preprocessing.py \
    --mosmed /kaggle/input \
    --out {PROCESSED_DIR} \
    --size 256 \
    --context 1 \
    --labeled-ratio 0.4


/kaggle/working/fyp-dtcr-unet/dtcr-unet
Found 50 paired MosMed volumes.
Total annotated studies with lesions: 50
Split breakdown: Train=30 studies, Val=10 studies, Test=10 studies.
Processing test split: 100%|████████████████████| 10/10 [00:16<00:00,  1.68s/it]

Preprocessing complete!
{
  "dataset": "MosMedData",
  "target_size": [
    256,
    256
  ],
  "total_extracted_slices": 1007,
  "splits": {
    "train": 640,
    "val": 181,
    "test": 186
  },
  "semi_supervised_breakdown": {
    "train_labeled": 227,
    "train_unlabeled": 413
  },
  "labeled_ratio": 0.4,
  "seed": 42
}


In [4]:
%cd /kaggle/working/fyp-dtcr-unet/dtcr-unet

!python train.py \
    --data /kaggle/working/data/processed \
    --epochs 80 \
    --batch-size 8 \
    --grad-accum 2 \
    --lr 0.001 \
    --labeled-ratio 0.4 \
    --checkpoints /kaggle/working/checkpoints \
    --outputs /kaggle/working/outputs


/kaggle/working/fyp-dtcr-unet/dtcr-unet
Using device: cuda | AMP Mixed Precision: True
Data ready: 640 train samples, 181 val samples.

Starting DTCR-U-Net training for 80 epochs...

Epoch 1/80 [Train]: 100%|█| 64/64 [01:04<00:00,  1.01s/it, Loss=1.5698, Seg=1.26
Epoch 01/80 (72.4s) | Train Loss: 1.8376 | Val Dice: 20.44% | Val F1: 20.44% | Val HD: 288.03px | (Best!)
Epoch 2/80 [Train]: 100%|█| 64/64 [01:04<00:00,  1.01s/it, Loss=1.2261, Seg=1.10
Epoch 02/80 (72.4s) | Train Loss: 1.4795 | Val Dice: 20.44% | Val F1: 20.44% | Val HD: 288.03px | 
Epoch 3/80 [Train]: 100%|█| 64/64 [01:03<00:00,  1.00it/s, Loss=1.1222, Seg=0.99
Epoch 03/80 (71.9s) | Train Loss: 1.3634 | Val Dice: 35.06% | Val F1: 35.06% | Val HD: 157.13px | (Best!)
Epoch 4/80 [Train]: 100%|█| 64/64 [01:03<00:00,  1.00it/s, Loss=1.5376, Seg=1.01
Epoch 04/80 (72.0s) | Train Loss: 1.2717 | Val Dice: 21.78% | Val F1: 21.78% | Val HD: 155.74px | 
Epoch 5/80 [Train]: 100%|█| 64/64 [00:58<00:00,  1.10it/s, Loss=1.0919, Seg=0.76
Ep

In [5]:
%cd /kaggle/working/fyp-dtcr-unet/dtcr-unet

!python evaluate.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pth \
    --data /kaggle/working/data/processed \
    --out /kaggle/working/outputs/evaluation \
    --batch-size 8


/kaggle/working/fyp-dtcr-unet/dtcr-unet
Loading checkpoint from: /kaggle/working/checkpoints/best_model.pth
Test samples: 186
Testing: 100%|██████████████████████████████████| 24/24 [00:33<00:00,  1.41s/it]

           TEST SET EVALUATION RESULTS           
DICE           : 27.24 ± 25.79
SENSITIVITY    : 79.98 ± 24.30
SPECIFICITY    : 99.38 ± 0.71
ACCURACY       : 99.31 ± 0.71
F1             : 27.24 ± 25.79
HD             : 161.71 ± 125.45
